# Hyperparameter tuning — scratch ResNet (Kaggle)

Runs Grid Search, Random Search and Optuna for the scratch ResNet, then compares them. Details: `HYPERPARAMETER_TUNING.md`.

**Notebook settings (right-hand panel):**
1. **Accelerator: GPU T4 x2.** Avoid P100: recent PyTorch builds no longer support it. The script uses one GPU.
2. **Internet: On.** Needed to clone the repo and download the CUB images from Hugging Face; Kaggle requires a phone-verified account for this.

**To run without keeping the browser open:** click **Save Version → Save & Run All (Commit)**. Results appear under the version's **Output** tab (`hyperparameter_tuning/`).

Kaggle stops a session after 12 hours, so the run is capped at 11 hours (`MAX_HOURS`). It stops starting new trials before that and still writes the comparison. If it didn't finish, attach this notebook's previous output as input (**Add Input → Your Work → Notebook Output Files**) and run it again: finished trials are copied in and skipped.

In [ ]:
REPO_URL = "https://github.com/TokaElsaayed/Deep-project.git"
BRANCH = "hyperparameter-tuning"
REPO_DIR = "/tmp/Deep-project"            # code lives outside /kaggle/working so it isn't saved as output
OUTPUT_DIR = "/kaggle/working/hyperparameter_tuning"
MAX_HOURS = 11
METHOD = "all"                            # "grid", "random", "optuna", "all" or "compare"

In [ ]:
!nvidia-smi
import torch
print(torch.__version__, "CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Turn on a GPU accelerator (GPU T4 x2) in the notebook settings."

In [ ]:
!rm -rf {REPO_DIR}
!git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log -1 --oneline

In [ ]:
# torch/torchvision are preinstalled on Kaggle; only add what may be missing (don't reinstall torch).
!pip install -q optuna datasets

## Resume from a previous run (optional)

If an earlier version's output is attached as input, copy its finished trials so they are skipped.

In [ ]:
import glob, os, shutil
os.makedirs(OUTPUT_DIR, exist_ok=True)
previous = sorted({os.path.dirname(p) for pattern in ("*_trials.csv", "optuna_study.db")
                   for p in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)})
for folder in previous:
    for name in os.listdir(folder):
        if name.endswith("_trials.csv") or name == "optuna_study.db":
            shutil.copy(os.path.join(folder, name), OUTPUT_DIR)
            print("Resuming from", os.path.join(folder, name))
if not previous:
    print("No previous results attached; starting fresh.")

In [ ]:
# Lists the grid and random-search settings that will be trained.
!python src/training/tune_scratch_resnet.py --method {METHOD} --dry-run

## Run the tuning

Each trial prints its validation accuracy per epoch. Results are saved after every trial.

In [ ]:
!python src/training/tune_scratch_resnet.py --method {METHOD} --output-dir {OUTPUT_DIR} --max-hours {MAX_HOURS}

## Results

In [ ]:
from IPython.display import Image, Markdown, display
!python src/training/tune_scratch_resnet.py --method compare --output-dir {OUTPUT_DIR}
display(Markdown(open(f"{OUTPUT_DIR}/comparison.md").read()))
display(Image(f"{OUTPUT_DIR}/best_score_comparison.png"))
display(Image(f"{OUTPUT_DIR}/search_space_coverage.png"))
print(open(f"{OUTPUT_DIR}/best_hyperparameters.json").read())